# Kafka multi-run comparison (matches your column names)

This notebook matches the metrics columns produced by your existing analysis notebooks, e.g.:
- `cluster_agg_*.csv` columns like: `throughput`, `p99_max`, `p99_med`, `viol_rate`, `skew_ratio`
- `per_pod_merged_*.csv` columns like: `pod`, `lat_p99_ms`, `viol_rate`, `lag_skew_ratio`, `msg_rate_window`, `cpu_percent`, `mem_percent`

It also supports the raw nested layout by recursively scanning `consumer_metrics*.csv` if the merged files are not present.

Outputs:
- `comparison_out/comparison_summary.csv`
- PDF comparison plots in `comparison_out/`


In [1]:
from __future__ import annotations
import re
from pathlib import Path
from typing import Any, Optional, Dict, List, Tuple

import pandas as pd
import matplotlib.pyplot as plt

# =========================
# USER SETTINGS
# =========================
RESULTS_ROOT = Path('../results').resolve()   
OUT_DIR = Path('./comparison_out').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Only used for plot label (your CSV already has viol_rate)
SLA_MS = 800

print('RESULTS_ROOT =', RESULTS_ROOT)
print('OUT_DIR      =', OUT_DIR)


RESULTS_ROOT = /Users/soheila/Desktop/RealTime-Streaming-Pipeline/results
OUT_DIR      = /Users/soheila/Desktop/RealTime-Streaming-Pipeline/analysis/comparison_out


In [2]:
# =========================
# Helpers
# =========================
def _safe_float(x: Any) -> Optional[float]:
    try:
        if x is None:
            return None
        if isinstance(x, str) and x.strip() == '':
            return None
        return float(x)
    except Exception:
        return None

def _pick_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    cols = list(df.columns)
    colset = set(cols)
    for c in candidates:
        if c in colset:
            return c
    lower_map = {c.lower(): c for c in cols}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None

def read_configmap_params(cfg_path: Path) -> Dict[str, Any]:
    """Read TRAFFIC_MODE, SKEW_FRACTION, SKEW_PARTITIONS from ConfigMap YAML.
    Minimal parsing (no PyYAML). Values are expected under `data:`.
    """
    params = {'traffic_mode': 'unknown', 'skew_fraction': None, 'skew_partitions': None}
    if not cfg_path.exists():
        return params

    in_data = False
    for raw in cfg_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = raw.rstrip('\n')

        if re.match(r'^\s*data\s*:\s*$', line):
            in_data = True
            continue
        if in_data and re.match(r'^\S', line):
            in_data = False
        if not in_data:
            continue

        m = re.match(r'^\s*([A-Za-z0-9_]+)\s*:\s*(.*)\s*$', line)
        if not m:
            continue
        k = m.group(1).strip()
        v = m.group(2).strip().strip('"').strip("'")

        if k == 'TRAFFIC_MODE':
            params['traffic_mode'] = (v or 'unknown').lower()
        elif k in ('SKEW_FRACTION', 'SKEW_FRAC', 'SKEWFRAC'):
            params['skew_fraction'] = _safe_float(v)
        elif k in ('SKEW_PARTITIONS', 'SKEW_PARTITION', 'SKEW_PARTS'):
            nums = re.findall(r'-?\d+', v)
            params['skew_partitions'] = ','.join(nums) if nums else v

    return params

def extract_pod_from_path(p: Path) -> str:
    s = str(p)
    m = re.search(r'(consumer-sts-\d+)', s)
    return m.group(1) if m else 'unknown'


In [7]:
# =========================
# Inputs: prefer merged files if present, else raw recursive metrics
# =========================
def find_inputs(consumer_dir: Path) -> Tuple[Optional[Path], Optional[Path]]:
    """Return (cluster_agg_csv, per_pod_merged_csv) if present (anywhere under consumer_dir)."""
    cluster = sorted(consumer_dir.rglob('cluster_agg_*.csv'))
    per_pod = sorted(consumer_dir.rglob('per_pod_merged_*.csv'))
    return (cluster[0] if cluster else None, per_pod[0] if per_pod else None)

def load_raw_consumer_metrics(consumer_dir: Path) -> pd.DataFrame:
    """Recursively load `consumer_metrics*.csv` and add `pod` if missing."""
    files = sorted(consumer_dir.rglob('consumer_metrics*.csv'))
    if not files:
        return pd.DataFrame()

    frames = []
    for f in files:
        try:
            df = pd.read_csv(f)
        except Exception as e:
            print(f'[WARN] failed reading {f}: {e}')
            continue

        # normalize pod
        if 'pod' not in df.columns:
            df['pod'] = extract_pod_from_path(f)
        df['__source_file'] = str(f)
        frames.append(df)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


In [8]:
# =========================
# Column names that match your produced CSVs
# =========================
# cluster_agg_*.csv
CL_THR_CANDS  = ['throughput']
CL_P99_MED    = ['p99_med']
CL_P99_MAX    = ['p99_max']
CL_VIO_CANDS  = ['viol_rate', 'sla_violation_rate', 'violation_rate']
CL_SKEW_CANDS = ['skew_ratio', 'lag_skew_ratio', 'skew_ratio_avg']

# per_pod_merged_*.csv
POD_THR_CANDS  = ['msg_rate_window', 'throughput', 'throughput_msg_s', 'throughput_msgs_s']
POD_P99_CANDS  = ['lat_p99_ms', 'p99_e2e_ms', 'p99_ms', 'true_p99_ms']
POD_VIO_CANDS  = ['viol_rate', 'sla_violation_rate', 'violation_rate']
POD_SKEW_CANDS = ['lag_skew_ratio', 'skew_ratio']
POD_CPU_CANDS  = ['cpu_percent', 'cpu']
POD_MEM_CANDS  = ['mem_percent', 'memory_percent', 'mem']
POD_PART_CANDS = ['partition', 'partition_id', 'topic_partition', 'tp']


In [9]:
# =========================
# Summarize one run
# =========================
def summarize_run(run_dir: Path) -> Dict[str, Any]:
    consumer_dir = run_dir / 'consumer'
    if not consumer_dir.exists():
        return {}

    cfg = read_configmap_params(consumer_dir / 'pipeline-configmap.yaml')
    cluster_csv, per_pod_csv = find_inputs(consumer_dir)

    df_cluster = pd.read_csv(cluster_csv) if cluster_csv else None
    df_pod = pd.read_csv(per_pod_csv) if per_pod_csv else None

    # Fallback: raw recursive files
    df_raw = None
    if df_cluster is None and df_pod is None:
        df_raw = load_raw_consumer_metrics(consumer_dir)

    out: Dict[str, Any] = {
        'run_id': run_dir.name,
        'traffic_mode': cfg.get('traffic_mode', 'unknown'),
        'skew_fraction': cfg.get('skew_fraction', None),
        'skew_partitions': cfg.get('skew_partitions', None),
    }

    # ----- Throughput -----
    thr = None
    if df_cluster is not None:
        c = _pick_col(df_cluster, CL_THR_CANDS)
        if c:
            thr = pd.to_numeric(df_cluster[c], errors='coerce').dropna().mean()
    if thr is None and df_pod is not None:
        c = _pick_col(df_pod, POD_THR_CANDS)
        if c:
            thr = pd.to_numeric(df_pod[c], errors='coerce').dropna().mean()
    if thr is None and df_raw is not None and not df_raw.empty:
        c = _pick_col(df_raw, POD_THR_CANDS + CL_THR_CANDS)
        if c:
            thr = pd.to_numeric(df_raw[c], errors='coerce').dropna().mean()
    out['throughput_avg_msg_s'] = float(thr) if thr is not None else None

    # ----- SLA violation -----
    vio = None
    if df_cluster is not None:
        c = _pick_col(df_cluster, CL_VIO_CANDS)
        if c:
            vio = pd.to_numeric(df_cluster[c], errors='coerce').dropna().mean()
    if vio is None and df_pod is not None:
        c = _pick_col(df_pod, POD_VIO_CANDS)
        if c:
            vio = pd.to_numeric(df_pod[c], errors='coerce').dropna().mean()
    if vio is None and df_raw is not None and not df_raw.empty:
        c = _pick_col(df_raw, POD_VIO_CANDS + CL_VIO_CANDS)
        if c:
            vio = pd.to_numeric(df_raw[c], errors='coerce').dropna().mean()
    out['sla_violation_avg'] = float(vio) if vio is not None else None

    # ----- Skew ratio -----
    sk = None
    if df_cluster is not None:
        c = _pick_col(df_cluster, CL_SKEW_CANDS)
        if c:
            sk = pd.to_numeric(df_cluster[c], errors='coerce').dropna().mean()
    if sk is None and df_pod is not None:
        c = _pick_col(df_pod, POD_SKEW_CANDS)
        if c:
            sk = pd.to_numeric(df_pod[c], errors='coerce').dropna().mean()
    if sk is None and df_raw is not None and not df_raw.empty:
        c = _pick_col(df_raw, POD_SKEW_CANDS + CL_SKEW_CANDS)
        if c:
            sk = pd.to_numeric(df_raw[c], errors='coerce').dropna().mean()
    out['skew_ratio_avg'] = float(sk) if sk is not None else None

    # ----- p99 + hot consumer -----
    out['p99_pods_median_ms'] = None
    out['p99_pods_max_ms'] = None
    out['hot_consumer'] = None
    out['hot_consumer_p99_mean_ms'] = None

    # Prefer per_pod_merged (it has pod + lat_p99_ms)
    df_for_p99 = df_pod if df_pod is not None else df_raw
    if df_for_p99 is not None and not df_for_p99.empty:
        p99_col = _pick_col(df_for_p99, POD_P99_CANDS + CL_P99_MED + CL_P99_MAX)
        if p99_col:
            df_for_p99['_p99'] = pd.to_numeric(df_for_p99[p99_col], errors='coerce')
            g = df_for_p99.groupby('pod', dropna=True)['_p99'].mean().dropna()
            if len(g) > 0:
                out['p99_pods_median_ms'] = float(g.median())
                out['p99_pods_max_ms'] = float(g.max())
                out['hot_consumer'] = str(g.idxmax())
                out['hot_consumer_p99_mean_ms'] = float(g.max())

    # ----- hot partition (only if partition exists) -----
    out['hot_partition'] = None
    out['hot_partition_p99_mean_ms'] = None
    if df_for_p99 is not None and not df_for_p99.empty and '_p99' in df_for_p99.columns:
        part_col = _pick_col(df_for_p99, POD_PART_CANDS)
        if part_col:
            gp = df_for_p99.groupby(part_col, dropna=True)['_p99'].mean().dropna()
            if len(gp) > 0:
                out['hot_partition'] = str(gp.idxmax())
                out['hot_partition_p99_mean_ms'] = float(gp.max())

    # ----- hot CPU consumer (optional) -----
    out['hot_cpu_consumer'] = None
    out['hot_cpu_percent_mean'] = None
    if df_for_p99 is not None and not df_for_p99.empty:
        cpu_col = _pick_col(df_for_p99, POD_CPU_CANDS)
        if cpu_col:
            df_for_p99['_cpu'] = pd.to_numeric(df_for_p99[cpu_col], errors='coerce')
            gcpu = df_for_p99.groupby('pod', dropna=True)['_cpu'].mean().dropna()
            if len(gcpu) > 0:
                out['hot_cpu_consumer'] = str(gcpu.idxmax())
                out['hot_cpu_percent_mean'] = float(gcpu.max())

    return out


In [13]:
# =========================
# Scan all runs + build summary
# =========================
run_dirs = sorted([p for p in RESULTS_ROOT.iterdir() if p.is_dir()])

rows = []
for rd in run_dirs:
    if not (rd / 'consumer').exists():
        continue
    row = summarize_run(rd)
    if row:
        rows.append(row)

summary = pd.DataFrame(rows)
if summary.empty:
    raise RuntimeError(f'No runs found under {RESULTS_ROOT}/*/consumer')

def mk_label(r: pd.Series) -> str:
    tm = r.get('traffic_mode') or 'na'
    sf = r.get('skew_fraction')
    sp = r.get('skew_partitions')
    sf_s = f"{sf:.2f}" if isinstance(sf, (int, float)) else 'na'
    sp_s = str(sp) if sp is not None else 'na'
    return f"{r['run_id']} | tm={tm} sf={sf_s} sp={sp_s}"

summary['label'] = summary.apply(mk_label, axis=1)

out_csv = OUT_DIR / 'comparison_summary.csv'
summary.to_csv(out_csv, index=False)
print('[OK] wrote', out_csv)

summary

[OK] wrote /Users/soheila/Desktop/RealTime-Streaming-Pipeline/analysis/comparison_out/comparison_summary.csv


,run_id,traffic_mode,skew_fraction,skew_partitions,throughput_avg_msg_s,sla_violation_avg,skew_ratio_avg,p99_pods_median_ms,p99_pods_max_ms,hot_consumer,hot_consumer_p99_mean_ms,hot_partition,hot_partition_p99_mean_ms,hot_cpu_consumer,hot_cpu_percent_mean,label
0,20260223_232437_skew,skew,0.3,0,34.605860,None,3.364783,1019.569065,288646.962763,consumer-sts-0,288646.962763,None,None,consumer-sts-1,47.974194,20260223_232437_skew | tm=skew sf=0.30 sp=0
1,20260223_235807_skew,skew,0.8,0,18.228813,None,3.838583,1612.784038,503018.385302,consumer-sts-3,503018.385302,None,None,consumer-sts-1,49.066038,20260223_235807_skew | tm=skew sf=0.80 sp=0
2,20260224_003023_balanced,balanced,NaN,,36.752174,None,2.790298,886.504457,932.104261,consumer-sts-3,932.104261,None,None,consumer-sts-1,48.613043,20260224_003023_balanced | tm=balanced sf=nan sp=


In [12]:
# =========================
# 'Hot' report table
# =========================
cols = [
    'run_id','traffic_mode','skew_fraction','skew_partitions',
    'throughput_avg_msg_s','p99_pods_median_ms','p99_pods_max_ms',
    'sla_violation_avg','skew_ratio_avg',
    'hot_consumer','hot_consumer_p99_mean_ms',
    'hot_cpu_consumer','hot_cpu_percent_mean',
    'hot_partition','hot_partition_p99_mean_ms'
]
cols = [c for c in cols if c in summary.columns]
summary[cols].sort_values(['traffic_mode','skew_fraction'], na_position='last')

,run_id,traffic_mode,skew_fraction,skew_partitions,throughput_avg_msg_s,p99_pods_median_ms,p99_pods_max_ms,sla_violation_avg,skew_ratio_avg,hot_consumer,hot_consumer_p99_mean_ms,hot_cpu_consumer,hot_cpu_percent_mean,hot_partition,hot_partition_p99_mean_ms
2,20260224_003023_balanced,balanced,NaN,,36.752174,886.504457,932.104261,None,2.790298,consumer-sts-3,932.104261,consumer-sts-1,48.613043,None,None
0,20260223_232437_skew,skew,0.3,0,34.605860,1019.569065,288646.962763,None,3.364783,consumer-sts-0,288646.962763,consumer-sts-1,47.974194,None,None
1,20260223_235807_skew,skew,0.8,0,18.228813,1612.784038,503018.385302,None,3.838583,consumer-sts-3,503018.385302,consumer-sts-1,49.066038,None,None
